# Overview

This code shows how to analyze/plot results from the Castro simulations, using the class `CastroSimulation` defined in `analysis_tool.py`.

In [ ]:
import numpy as np
import os
import tqdm
import matplotlib.pyplot as plt
from scipy.constants import m_p, e
from scipy.interpolate import RegularGridInterpolator
import sys
from analysis_tool import CastroSimulation
import re
import glob
import subprocess
sys.path.append("../../initial_condition")
from ionization_routines import save_to_openpmd
%config InlineBackend.figure_format = 'svg'

# Tutorial 2D rz #

### Generate inputs for the simulation ###

In [ ]:
sim_folder = '../run/run_tutorial_2d/'
os.makedirs(sim_folder, exist_ok=True)

r = np.linspace(0.0, 600e-6, 256)
z = np.linspace(0.0, 600e-6, 256)

R, Z = np.meshgrid(r, z, indexing='ij')
sigma = 30e-6
T_peak = 1000.0  # eV
T_min = 1e-3     # eV, small temperature floor
T_eV = T_min + (T_peak - T_min) * np.exp(- R**2 / sigma**2) # Gaussian profile of temperature

# Species keys
with open('../build/species.net', 'r') as f:
    species_keys = re.findall(r'\n\s.*\s([A-Z][a-z]*\d)', f.read())

# Populations array
populations = np.zeros((R.shape[0], Z.shape[1], len(species_keys)))
populations[:, :,  species_keys.index('H1')] = 1.0 - 1e-3
populations[:, :, species_keys.index('H0')] = 1e-3
# Save file
save_to_openpmd({'r': [r.min(), r.max()], 'z': [z.min(), z.max()]},
            populations, T_eV, f'{sim_folder}2d_inputs.h5', species_keys)

### Run simulation ###

In [ ]:
def run_castro_simulation(runtime_options):
    """
    Run the Castro simulation directly (without srun) in a specific folder.
    """
    # Find the Castro executable (absolute path)
    build_dir = os.path.abspath("../build")
    executables = glob.glob(os.path.join(build_dir, "Castro2d*"))
    if len(executables) == 0:
        raise FileNotFoundError(f"No Castro2d executable found in {build_dir}")
    elif len(executables) > 1:
        raise RuntimeError(f"Multiple Castro2d executables found: {executables}")
    executable = os.path.abspath(executables[0])

    # Absolute path for the inputs file
    inputs = os.path.abspath("../run/inputs.2d.cyl")

    # Create the simulation folder if it doesn't exist
    os.makedirs(sim_folder, exist_ok=True)

    # Build the command
    command = f"{executable} {inputs} {runtime_options}"

    # Run Castro inside sim_folder
    try:
        subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            check=True,
            cwd=sim_folder
        )
    except subprocess.CalledProcessError as e:
        print(f"Command failed with exit code {e.returncode}")
        print("STDOUT:", e.stdout)
        print("STDERR:", e.stderr)
        raise

run_castro_simulation(runtime_options=f"castro.add_ext_src = 0 amr.max_level = 2 castro.diffuse_temp = 0 amr.max_grid_size = 128 problem.initial_conditions_file=2d_inputs.h5")

### Analysis ###

In [ ]:
run_dir =  sim_folder
file_start = 'plt_2d_'

cs = CastroSimulation(run_dir, file_start)
cs.sim_info()

### Plot fields ###

In [ ]:
t = cs.output_times[10]
m = cs.get_field(t, quantity='density', level = 2)

plt.figure(figsize=(6,5))
plt.imshow(m['q'], extent=[m['z'][0], m['z'][-1], m['r'][0], m['r'][-1]], origin='lower')
plt.colorbar()
plt.title('2D Density Profile')
plt.xlabel('r')
plt.ylabel('z')

m_slice = cs.get_field(t, quantity='density', level = 2, positions = {'z':0.03})

plt.figure(figsize=(6,4))
plt.plot(m_slice['r'], m_slice['q'])
plt.title('Density Profile at z=0.03 cm')
plt.xlabel('r')
plt.ylabel('Density')


### Plot energies ###

In [ ]:
# ---- Single time energy calculation ----

t_single = cs.output_times[10]
energy_single = cs.get_energy(t_single, level=2)[0]
print(f"Total energy at time {t_single} s: {energy_single} erg")

# ---- Time array energy calculation ----

t_array = cs.output_times
energy_array = cs.get_energy(t_array, level=2)[0]

thermal_energy_array = cs.get_energy(t_array, level=2, energy_type='thermal')[0]
kinetic_energy_array = cs.get_energy(t_array, level=2, energy_type='kinetic')[0]

plt.figure()
plt.plot(t_array, thermal_energy_array, label='Thermal Energy')
plt.plot(t_array, kinetic_energy_array, label='Kinetic Energy')
plt.plot(t_array, energy_array, label='Total Energy', linestyle='--', color='black')
plt.xlabel('Time (s)')
plt.ylabel('Total Energy (erg)')
plt.ylim(0, max(energy_array)*1.1)
plt.legend()

# Examples of usage #

### Load the data and show fields ###

In [ ]:
run_dir = '../run/test_1D/'
file_start = 'plt_1d_'

cs_1d = CastroSimulation(run_dir, file_start)
cs_1d.sim_info()

run_dir = '../run/test_2D_xy/'
file_start = 'plt_2d_'

cs_2d_xy = CastroSimulation(run_dir, file_start)

run_dir = '../run/test_2D_rz/'
file_start = 'plt_2d_'

cs_2d_rz = CastroSimulation(run_dir, file_start)

run_dir = '../run/test_3D/'
file_start = 'plt_3d_'
cs_3d = CastroSimulation(run_dir, file_start)

### Plot the field with the right coordinates  1D / 2D ###

In [ ]:
t_1d = cs_1d.output_times[10]
m_1d = cs_1d.get_field(t_1d, quantity='density', level=3)

plt.figure(figsize=(13,5))
plt.subplot(1, 3, 1)
plt.plot(m_1d['r'], m_1d['q'])
plt.title('1D Density Profile')
plt.xlabel('Radius')
plt.ylabel('Density')

t_2d = cs_2d_xy.output_times[10]
m_2d_xy = cs_2d_xy.get_field(t_2d, quantity='density', level=3)

plt.subplot(1, 3, 2)
plt.imshow(m_2d_xy['q'].T, extent=[m_2d_xy['x'][0], m_2d_xy['x'][-1], m_2d_xy['y'][0], m_2d_xy['y'][-1]], origin='lower')
plt.colorbar()
plt.title('2D Density Profile XY')
plt.xlabel('X')
plt.ylabel('Y')

t_2d_rz = cs_2d_rz.output_times[10]
m_2d_rz = cs_2d_rz.get_field(t_2d, quantity='density', level=3)

plt.subplot(1, 3, 3)
plt.imshow(m_2d_rz['q'], extent=[m_2d_rz['z'][0], m_2d_rz['z'][-1], m_2d_rz['r'][0], m_2d_rz['r'][-1]], origin='lower')
plt.colorbar()
plt.title('2D Density Profile RZ')
plt.xlabel('z')
plt.ylabel('r')

plt.tight_layout()


### Plot field 3D ###

### Make a slice of a 2D/3D fields ###

In [ ]:
t_slice = cs_2d_xy.output_times[10]
m_slice_y = cs_2d_xy.get_field(t_slice, quantity = 'density', positions = {'y' : 0.03}, level=3)
plt.figure(figsize=(10,5))
plt.subplot(1, 2, 1)
plt.plot(m_slice_y['y'], m_slice_y['q'])
plt.title('2D Density Slice')
plt.xlabel('Y')
plt.ylabel('Density')

m_slice_z = cs_2d_rz.get_field(t_slice, quantity = 'density', positions = {'z' : 0.06}, level=3)
plt.subplot(1, 2, 2)
plt.plot(m_slice_z['r'], m_slice_z['q'])
plt.title('2D Density Slice')
plt.xlabel('r')
plt.ylabel('Density')

plt.tight_layout()

### Plot the energy as function of time / for a single time ### 

In [ ]:
t_single = cs_1d.output_times[10]
energy_single = cs_1d.get_energy(t_single, level=3)
print(f"Total energy at time {t_single} s: {energy_single} erg")

t_array = cs_1d.output_times
energy_array = cs_1d.get_energy(t_array, level=3)[0]

thermal_energy_array = cs_1d.get_energy(t_array, level=3, energy_type='thermal')[0]
kinetic_energy_array = cs_1d.get_energy(t_array, level=3, energy_type='kinetic')[0]
ionization_energy_array = cs_1d.get_energy(t_array, level=3, energy_type='ion')[0]

plt.figure()
plt.plot(t_array, thermal_energy_array, label='Thermal Energy')
plt.plot(t_array, kinetic_energy_array, label='Kinetic Energy')
plt.plot(t_array, ionization_energy_array, label='Ionization Energy')
plt.plot(t_array, energy_array, label='Total Energy', linestyle='--', color='black')
plt.xlabel('Time (s)')
plt.ylabel('Total Energy (erg)')
plt.ylim(0, max(energy_array)*1.1)
plt.legend()

In [ ]:
t_single = cs_2d_rz.output_times[10]
energy_single = cs_2d_rz.get_energy(t_single, level=3)
print(f"Total energy at time {t_single} s: {energy_single} erg")

t_array = cs_2d_rz.output_times
energy_array = cs_2d_rz.get_energy(t_array, level=3)[0]

thermal_energy_array = cs_2d_rz.get_energy(t_array, level=3, energy_type='thermal')[0]
kinetic_energy_array = cs_2d_rz.get_energy(t_array, level=3, energy_type='kinetic')[0]

plt.figure()
plt.plot(t_array, thermal_energy_array, label='Thermal Energy')
plt.plot(t_array, kinetic_energy_array, label='Kinetic Energy')
plt.plot(t_array, energy_array, label='Total Energy', linestyle='--', color='black')
plt.xlabel('Time (s)')
plt.ylabel('Total Energy (erg)')
plt.ylim(0, max(energy_array)*1.1)
plt.legend()

### Get particle number for a given species ###

In [ ]:
t_single = cs_1d.output_times[10]
part_H1_single = cs_1d.get_particle_number(t_single, species='H1', level=3)[0]
part_H0_single = cs_1d.get_particle_number(t_single, species='H0', level=3)[0]

print(f"Number of H1 particles at time {t_single} s: {part_H1_single:.2e}")
print(f"Number of H0 particles at time {t_single} s: {part_H0_single:.2e}")

t_array = cs_1d.output_times
part_H1_array = cs_1d.get_particle_number(t_array, species='H1', level=3)[0]
part_H0_array = cs_1d.get_particle_number(t_array, species='H0', level=3)[0]

plt.figure()
plt.plot(t_array, part_H1_array, label='H1 Particle Number')
plt.plot(t_array, part_H0_array, label='H0 Particle Number')
plt.xlabel('Time (s)')
plt.ylabel('Particle Number')
plt.yscale('log')
plt.legend()


# Compare data with theory

In [ ]:
sys.path.append('../../theory/sedov_theory/python/')
from sedov_theory import SedovTalorProblem

# Calculate analytical solution
gamma = 5./3
rho = 1.67e-6 # g / cm^3
E = 4400 # erg / cm

sol = SedovTalorProblem(gamma, E, rho)

In [ ]:
plt.figure(figsize=(5, 4))
plt.clf()
level = 3
time = 4e-9

labels = {
    'rho_H0': 'Density of Hydrogen atoms',
    'rho_H1': 'Density of Hydrogen ions',
    'density': 'Total Hydrogen density',
}
colors = {
    'rho_H0': 'gray',
    'rho_H1': 'blue',
    'density': 'black'
}

for quantity in ['density', 'rho_H0', 'rho_H1']:
    r, q, t = cs.extract_data( time, quantity, level)
    # Convert r from cm to microns
    # Convert massic density in cc to number density in cc
    plt.plot(1e4*r, q/1.67e-6, '-o', label=labels[quantity], color=colors[quantity])

# Plot Sedov-Taylor solution
r_th = r
plt.plot( 1e4*r_th, sol.evaluate( 'density', r, t )/1.67e-6, 'k--', label='Sedov-Taylor theory' )
plt.legend(loc=1)

plt.ylabel('Density (10$^{18}$ cm$^{-3}$)')
plt.xlabel('r ($\mu m$)')
plt.title(f'Density at t={t*1.e9:.1f} ns')

In [ ]:
# plot data at difference resolutions
quantity = 'pressure'

plt.clf()
for level in range(3,-1,-1):
    r, q, t = cs.extract_data( 1.e-9, quantity, level)
    plt.plot(r, q, '-o', label='level=%d'%level)
    if level == 3:
        r_th = r
        t_th = t
#plt.plot( r_th, sol.evaluate( quantity, r_th, t_th ), 'k--', label='analytical' )
plt.legend(loc=0)

In [ ]:
# plot data at difference resolutions
quantity = 'Temp'

plt.clf()
for level in range(3,-1,-1):
    r, q, t = cs.extract_data( 80, quantity, level)
    plt.plot(r, q, '-o', label='level=%d'%level)
    if level == 3:
        r_th = r
        t_th = t
#plt.plot( r_th, sol.evaluate( quantity, r_th, t_th ), 'k--', label='analytical' )
plt.legend(loc=0)

In [ ]:
# Extract data from different time
# Note that time is not regularly spaced
q_arr = []
rmax_arr = []
for time in tqdm.tqdm( cs.output_times ):
    r, q, t = cs.extract_data(time, 'density', level=2)
    rmax = r[np.argmax(q)]
    rmax_arr.append(rmax)
    q_arr.append(q)
q_arr = np.stack(q_arr)
t_arr = cs.output_times.copy()
r_arr = r

# Interpolate on a grid with regularly-spaced time
interp = RegularGridInterpolator(points=(t_arr, r_arr), values=q_arr, bounds_error=False, fill_value=None)
t_interp, r_interp = np.meshgrid(
    np.linspace(0, t_arr.max(), 1000),
    np.linspace(0, r_arr.max(), 1000), indexing='ij')
q_interp = interp((t_interp, r_interp))

In [ ]:
plt.figure(figsize=(6, 2))
plt.imshow(q_interp.T/1.67e-6, origin='lower',
           extent=[0, t_arr.max()*1.e9, 0, r_arr.max()*1.e4],
           aspect='auto', vmax=3)
cb = plt.colorbar()
cb.set_label('H density (10$^{18}$ cm$^{-3}$)')
#plt.plot(t_arr, rmax_arr, 'r-')
r_analytical = sol.blast_radius(t_arr)
#plt.plot(t_arr*1.e9, r_analytical*1.e4, 'r--', label='Sedov-Taylor theory')
#plt.legend(loc=0)
plt.ylabel('r ($\mu m$)')
plt.xlabel('t (ns)')
plt.ylim(0,300)